# Top-10 disparity features per category -> 2x2 heatmap grids

For each of the 12 (race x gender) categories, on block `down.2.1`:
1. Compute freq/mag over all `N_SEEDS` seeds (same recipe as `sae_race_feature_analysis.ipynb`,
   but at race+gender granularity instead of race-only), and rank features by
   `disparity = freq[category] - mean(freq[other 11 categories])`.
2. Pick 4 random seeds for that category, regenerate them to get spatial SAE feature maps.
3. For each of the top-10 features, render a 2x2 grid of the 4 images with that feature's
   activation heatmap overlaid (same jet/alpha-composite recipe as `sdxl-unbox/app.py`).

Outputs: `baseline/heatmaps/top10_by_category/<category>/feature<idx>_rank<r>_grid.png`
plus `baseline/heatmaps/top10_by_category/top10_summary.csv`.

In [1]:
import os
os.chdir('/n/fs/goose/ReNO')

import sys
sys.path.append('/n/fs/goose/sdxl-unbox')

import re
import csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
from pytorch_lightning import seed_everything

from diffusers import AutoencoderKL, EulerAncestralDiscreteScheduler
from SDLens import HookedStableDiffusionXLPipeline
from SAE import SparseAutoencoder

def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.2.2+cu121)
    Python  3.10.14 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


In [2]:
# Config
CACHE_DIR = 'hf_cache'
N_INFERENCE_STEPS = 1
DTYPE = torch.float16
DEVICE = torch.device('cuda')

CODE = 'down.2.1'
CODE_TO_BLOCK = {
    "down.2.1": "unet.down_blocks.2.attentions.1",
    "mid.0": "unet.mid_block.attentions.0",
    "up.0.0": "unet.up_blocks.0.attentions.0",
    "up.0.1": "unet.up_blocks.0.attentions.1",
}
CHECKPOINT_DIR = '/n/fs/goose/sdxl-unbox/checkpoints'
OUT_DIR = '/n/fs/goose/baseline/heatmaps/top10_by_category'
os.makedirs(OUT_DIR, exist_ok=True)

RACES = ["black", "white", "east asian", "middle eastern", "indian", "hispanic"]
GENDERS = ["man", "woman"]
CATEGORIES = [(race, gender) for gender in GENDERS for race in RACES]  # 12 categories

PROMPT_TEMPLATE = (
    "a portrait of a {race} {gender} with a neutral expression, "
    "without any facial coverings, in a neutral white colored t-shirt against a neutral white background"
)

N_SEEDS = 50          # seeds used to compute stable disparity stats per category
N_RANDOM_SEEDS = 4    # seeds used for the 2x2 heatmap grid
TOP_K = 10
RNG_SEED = 0          # for picking the 4 random seeds per category, reproducibly

def cat_slug(race, gender):
    return slugify(f"{race}_{gender}")

In [3]:
hooked_vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE, cache_dir=CACHE_DIR,
)
hooked_pipe = HookedStableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", vae=hooked_vae, torch_dtype=DTYPE, variant="fp16",
    use_safetensors=True, cache_dir=CACHE_DIR,
)
hooked_pipe.pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    hooked_pipe.pipe.scheduler.config, timestep_spacing="trailing",
)
hooked_pipe.pipe = hooked_pipe.pipe.to(DEVICE, DTYPE)

sae = SparseAutoencoder.load_from_disk(
    os.path.join(CHECKPOINT_DIR, f"{CODE_TO_BLOCK[CODE]}_k10_hidden5120_auxk256_bs4096_lr0.0001", "final")
).to(DEVICE)
n_dirs = sae.n_dirs
print("n_dirs:", n_dirs)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:  57%|█████▋    | 4/7 [00:00<00:00, 32.93it/s]

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  6.35it/s]

n_dirs: 5120


In [4]:
def get_diff_and_image(prompt, seed):
    """Replays the deterministic (seed_everything, latents, generator) recipe used
    throughout baseline/, returning the generated image and the block's residual
    delta (output - input), shaped [h, w, d_model]."""
    seed_everything(seed)
    generator = torch.Generator("cuda").manual_seed(seed)
    latents = torch.randn((1, 4, 64, 64), device=DEVICE, dtype=DTYPE)

    with torch.no_grad():
        images, cache = hooked_pipe.run_with_cache(
            prompt,
            latents=latents,
            generator=generator,
            num_inference_steps=N_INFERENCE_STEPS,
            guidance_scale=0.0,
            positions_to_cache=[CODE_TO_BLOCK[CODE]],
            save_input=True,
            save_output=True,
        )
    image = images.images[0]

    block = CODE_TO_BLOCK[CODE]
    diff = cache["output"][block] - cache["input"][block]
    if diff.shape[0] == 2:  # classifier-free guidance batch: keep the conditional half
        diff = diff[1].unsqueeze(0)
    diff_last = diff[:, -1].permute(0, 2, 3, 1).squeeze(0)  # [h, w, d_model]
    return image, diff_last

In [5]:
# --- Step A: freq/mag stats over all N_SEEDS, per (race, gender) category ---
feature_active_count = {cat: torch.zeros(n_dirs) for cat in CATEGORIES}
feature_mag_sum = {cat: torch.zeros(n_dirs) for cat in CATEGORIES}
n_samples = {cat: 0 for cat in CATEGORIES}

for cat in CATEGORIES:
    race, gender = cat
    prompt = PROMPT_TEMPLATE.format(race=race, gender=gender)
    for seed in range(N_SEEDS):
        _, diff_last = get_diff_and_image(prompt, seed)
        h, w, d_model = diff_last.shape
        with torch.no_grad():
            feats = sae.encode(diff_last.reshape(-1, d_model).float().to(DEVICE))
        feature_active_count[cat] += (feats > 0).float().sum(dim=0).cpu()
        feature_mag_sum[cat] += feats.sum(dim=0).cpu()
        n_samples[cat] += 1
    print(f"done stats for category={cat}")

[rank: 0] Seed set to 0


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.74it/s]

100%|██████████| 1/1 [00:00<00:00,  2.74it/s]


[rank: 0] Seed set to 1


...

100%|██████████| 1/1 [00:00<00:00, 15.08it/s]


[rank: 0] Seed set to 49


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 15.09it/s]

done stats for category=('hispanic', 'woman')


In [6]:
# --- disparity[cat] = freq[cat] - mean(freq[other 11 categories]) ---
freq = torch.stack([feature_active_count[cat] / max(n_samples[cat], 1) for cat in CATEGORIES])
mag = torch.stack([feature_mag_sum[cat] / max(n_samples[cat], 1) for cat in CATEGORIES])
mean_others = (freq.sum(dim=0, keepdim=True) - freq) / (len(CATEGORIES) - 1)
disparity = freq - mean_others

top10_by_category = {}
summary_rows = []
for i, cat in enumerate(CATEGORIES):
    order = torch.argsort(disparity[i].abs(), descending=True)[:TOP_K]
    top10_by_category[cat] = order.tolist()
    for rank, feat_idx in enumerate(order.tolist()):
        summary_rows.append({
            "race": cat[0], "gender": cat[1], "rank": rank, "feature_idx": feat_idx,
            "freq": freq[i, feat_idx].item(), "mag": mag[i, feat_idx].item(),
            "disparity": disparity[i, feat_idx].item(),
        })

with open(os.path.join(OUT_DIR, "top10_summary.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["race", "gender", "rank", "feature_idx", "freq", "mag", "disparity"])
    writer.writeheader()
    writer.writerows(summary_rows)

for cat in CATEGORIES:
    print(cat, top10_by_category[cat])

('black', 'man') [527, 2707, 3894, 1584, 2611, 472, 1664, 3764, 1522, 2250]
('white', 'man') [2707, 1653, 3894, 2808, 1817, 1584, 3764, 2893, 527, 2611]
('east asian', 'man') [2707, 1653, 527, 1584, 1817, 472, 3764, 2611, 2055, 3894]
('middle eastern', 'man') [452, 1653, 2893, 1584, 669, 527, 3764, 2611, 4858, 464]
('indian', 'man') [1653, 1584, 2707, 2611, 3764, 452, 3894, 472, 1664, 2055]
('hispanic', 'man') [1653, 1678, 1584, 3894, 3764, 2611, 1817, 2055, 472, 2893]
('black', 'woman') [527, 3279, 2707, 1653, 3764, 2893, 3894, 4957, 2611, 472]
('white', 'woman') [1584, 2611, 472, 1653, 2250, 527, 3894, 2707, 433, 2055]
('east asian', 'woman') [1584, 1653, 2707, 3764, 527, 3894, 472, 2611, 2250, 1664]
('middle eastern', 'woman') [138, 472, 2707, 2670, 1653, 1584, 4544, 3894, 2611, 433]
('indian', 'woman') [2707, 1653, 1584, 3764, 3894, 2893, 2611, 2527, 2055, 2645]
('hispanic', 'woman') [2707, 1653, 1584, 2250, 3764, 3894, 2893, 2611, 1678, 472]


In [7]:
# --- Step B: 4 random seeds per category -> spatial feature maps ---
rng = np.random.default_rng(RNG_SEED)
category_samples = {}  # cat -> list of (image, feats[h, w, n_dirs])

for cat in CATEGORIES:
    race, gender = cat
    prompt = PROMPT_TEMPLATE.format(race=race, gender=gender)
    chosen_seeds = rng.choice(N_SEEDS, size=N_RANDOM_SEEDS, replace=False).tolist()

    samples = []
    for seed in chosen_seeds:
        image, diff_last = get_diff_and_image(prompt, seed)
        h, w, d_model = diff_last.shape
        with torch.no_grad():
            feats = sae.encode(diff_last.reshape(-1, d_model).float().to(DEVICE))
        feats = feats.reshape(h, w, n_dirs).cpu().numpy()
        samples.append((image, feats, seed))
    category_samples[cat] = samples
    print(f"category={cat} seeds={chosen_seeds}")

[rank: 0] Seed set to 13


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 14.90it/s]


[rank: 0] Seed set to 25


  0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00, 15.09it/s]


[rank: 0] Seed set to 17


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 15.09it/s]

category=('hispanic', 'woman') seeds=[25, 33, 43, 17]


In [8]:
def make_heatmap_overlay(image: Image.Image, feature_map: np.ndarray, upscale: int) -> Image.Image:
    """Same jet/alpha-composite recipe as sdxl-unbox/app.py:plot_image_heatmap."""
    heatmap = np.kron(feature_map, np.ones((upscale, upscale)))
    image = image.convert("RGBA")

    jet = plt.cm.jet
    cmap = jet(np.arange(jet.N))
    cmap[:1, -1] = 0
    cmap[1:, -1] = 0.6
    cmap = ListedColormap(cmap)

    denom = np.max(heatmap) - np.min(heatmap)
    heatmap = (heatmap - np.min(heatmap)) / denom if denom > 0 else np.zeros_like(heatmap)
    heatmap_rgba = cmap(heatmap)
    heatmap_image = Image.fromarray((heatmap_rgba * 255).astype(np.uint8)).resize(image.size)

    return Image.alpha_composite(image, heatmap_image)

In [9]:
# --- Step C: for each category's top-10 features, render a 2x2 grid ---
for cat in CATEGORIES:
    race, gender = cat
    cat_dir = os.path.join(OUT_DIR, cat_slug(race, gender))
    os.makedirs(cat_dir, exist_ok=True)
    samples = category_samples[cat]  # 4 x (image, feats[h, w, n_dirs], seed)

    for rank, feat_idx in enumerate(top10_by_category[cat]):
        fig, axes = plt.subplots(2, 2, figsize=(8, 9))
        for ax, (image, feats, seed) in zip(axes.flat, samples):
            feature_map = feats[:, :, feat_idx]
            upscale = image.size[0] // feature_map.shape[1]
            overlay = make_heatmap_overlay(image, feature_map, upscale)
            ax.imshow(overlay)
            ax.set_title(f"seed={seed}  max act={feature_map.max():.2f}", fontsize=9)
            ax.set_aspect("equal")
            ax.axis("off")
        fig.suptitle(
            f"{race} {gender} -- feature {feat_idx} ({CODE}), rank {rank+1}/{TOP_K}", y=1.02,
        )
        fig.tight_layout()
        out_path = os.path.join(cat_dir, f"feature{feat_idx}_rank{rank+1}_grid.png")
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)

    print(f"done grids for category={cat}")

done grids for category=('black', 'man')


done grids for category=('white', 'man')


done grids for category=('east asian', 'man')


done grids for category=('middle eastern', 'man')


done grids for category=('indian', 'man')


...

done grids for category=('white', 'woman')


done grids for category=('east asian', 'woman')


done grids for category=('middle eastern', 'woman')


done grids for category=('indian', 'woman')


done grids for category=('hispanic', 'woman')
